In [1]:
# !pip install -q transformers datasets scikit-learn torch accelerate

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import torch
from torch.utils.data import DataLoader, RandomSampler, WeightedRandomSampler
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

# ================== CONFIG ==================
DATA_PATH = "/kaggle/input/random-datset/blp25_hatespeech_subtask_1A_train.tsv"  # your upload
DATA_PATH_2= "/kaggle/input/random-datset/blp25_hatespeech_subtask_1A_dev.tsv"  # your uploaded file
TEXT_COL = "text"       # <-- change if needed (e.g., "tweet", "sentence")
LABEL_COL = "label"
MODEL_NAME = "csebuetnlp/banglabert"  # or "roberta-base"
MAX_LEN = 128
TEST_SIZE = 0.15
RANDOM_STATE = 42
BATCH_SIZE = 16
NUM_EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01

# Choose ROS strategy
USE_SAMPLER = False  # True = B. WeightedRandomSampler; False = A. Materialize balanced dataset
# ============================================

# ---------- Load data ----------
train_df = pd.read_csv(DATA_PATH, sep="\t", keep_default_na=False)
val_df = pd.read_csv(DATA_PATH_2, sep="\t", keep_default_na=False)

# Map string labels to ids if needed
if train_df[LABEL_COL].dtype == object:
    label2id = {lbl: i for i, lbl in enumerate(sorted(train_df[LABEL_COL].unique()))}
    id2label = {i: lbl for lbl, i in label2id.items()}
    train_df[LABEL_COL] = train_df[LABEL_COL].map(label2id)
    val_df[LABEL_COL] = val_df[LABEL_COL].map(label2id)
else:
    classes_sorted = sorted(train_df[LABEL_COL].unique().tolist())
    label2id = {int(k): int(k) for k in classes_sorted}
    id2label = {i: str(i) for i in range(len(classes_sorted))}

num_labels = len(label2id)

# ---------- Stratified split ----------
# train_df, val_df = train_test_split(
#     df, test_size=TEST_SIZE, stratify=df[LABEL_COL], random_state=RANDOM_STATE
# )

# ---------- A. ROS by materializing a balanced train set ----------
def make_balanced_copy(train_df):
    # Get per-class groups
    groups = [g for _, g in train_df.groupby(LABEL_COL)]
    counts = [len(g) for g in groups]
    max_count = max(counts)

    balanced_parts = []
    for g in groups:
        n = len(g)
        if n == max_count:
            balanced_parts.append(g)
        else:
            # sample with replacement to reach max_count
            idx = np.random.choice(g.index, size=max_count - n, replace=True)
            balanced_parts.append(pd.concat([g, train_df.loc[idx]], axis=0))
    balanced = pd.concat(balanced_parts, axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    return balanced

if not USE_SAMPLER:
    train_df = make_balanced_copy(train_df)

# ---------- Build HF datasets ----------
train_ds = Dataset.from_pandas(train_df[[TEXT_COL, LABEL_COL]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[[TEXT_COL, LABEL_COL]], preserve_index=False)
raw = DatasetDict({"train": train_ds, "validation": val_ds})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

tok = raw.map(tokenize, batched=True)
tok = tok.rename_column(LABEL_COL, "labels")
tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# ---------- Model ----------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()},
)

# ---------- Metrics ----------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": (preds == labels).mean(),
    }

# ---------- B. ROS via WeightedRandomSampler (optional) ----------
def build_weighted_sampler(dataset):
    # Compute class counts on the *original* (unbalanced) train_df
    labels = np.array(dataset["labels"])
    class_counts = np.bincount(labels, minlength=num_labels).astype(float)
    # sample weight for each class = inverse of its frequency
    class_weights = 1.0 / np.maximum(class_counts, 1.0)
    sample_weights = class_weights[labels]
    # length of an "epoch": match the original dataset size
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),  # or a multiple if you want longer epochs
        replacement=True,
    )
    return sampler

class ROSTrainer(Trainer):
    """Use WeightedRandomSampler for ROS when USE_SAMPLER=True."""
    def get_train_dataloader(self):
        if USE_SAMPLER:
            sampler = build_weighted_sampler(self.train_dataset)
            return DataLoader(
                self.train_dataset,
                batch_size=self.args.train_batch_size,
                sampler=sampler,
                collate_fn=self.data_collator,
                drop_last=self.args.dataloader_drop_last,
                num_workers=self.args.dataloader_num_workers,
                pin_memory=self.args.dataloader_pin_memory,
            )
        # default RandomSampler if not using sampler
        return super().get_train_dataloader()

# ---------- Train ----------
training_args = TrainingArguments(
    output_dir="./outputs_ros",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = ROSTrainer(
    model=model,
    args=training_args,
    train_dataset=tok["train"],
    eval_dataset=tok["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Training with ROS:", "WeightedRandomSampler" if USE_SAMPLER else "Materialized balanced dataset")
trainer.train()

# ---------- Evaluate ----------
preds = trainer.predict(tok["validation"])
y_true = preds.label_ids
y_pred = preds.predictions.argmax(axis=1)
target_names = [id2label[i] for i in range(num_labels)]
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))


2025-10-02 18:56:55.675861: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759431415.862813      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759431415.917817      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_36/346582216.py:166: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `ROSTrainer.__init__`. Use `processing_class` instead.
  trainer = ROSTrainer(


Training with ROS: Materialized balanced dataset


model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.298500,1.126356,0.493140,0.661624
2,0.227500,1.331919,0.512367,0.702627
3,0.085500,1.719297,0.494612,0.688694


                precision    recall  f1-score   support

       Abusive     0.5088    0.5621    0.5341       564
          None     0.8235    0.7877    0.8052      1451
Political Hate     0.5735    0.5498    0.5614       291
       Profane     0.7442    0.8153    0.7781       157
Religious Hate     0.3542    0.4474    0.3953        38
        Sexism     0.0000    0.0000    0.0000        11

      accuracy                         0.7026      2512
     macro avg     0.5007    0.5270    0.5124      2512
  weighted avg     0.7082    0.7026    0.7047      2512

